# AtliQ Bank Credit Intelligence
## Day 2: Exploratory Data Analysis (EDA)

### Objectives
- Inspect the three source datasets without modifying the raw files.
- Explore data distributions, missing values, and unusual observations.
- Compare observed default rates across relevant groups.
- Document findings and unresolved data-quality questions.

**Important limitation:** The target-generation logic and observation horizon are undocumented. Observed associations do not establish causation or prove future-default prediction.

In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Python environment:", __import__("sys").executable)
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Plotly:", px.__name__)

assert RAW_DATA_DIR.exists(), f"Dataset folder not found: {RAW_DATA_DIR}"


Project root: c:\Users\AQUASA AZIZ\Documents\atliq-credit-intelligence
Raw data directory: c:\Users\AQUASA AZIZ\Documents\atliq-credit-intelligence\data\raw
Python environment: c:\Users\AQUASA AZIZ\Documents\atliq-credit-intelligence\.venv\Scripts\python.exe
pandas: 3.0.6
NumPy: 2.5.4
Plotly: plotly.express


In [2]:

customers = pd.read_csv(RAW_DATA_DIR / "customers.csv")
loans = pd.read_csv(RAW_DATA_DIR / "loans.csv")
bureau = pd.read_csv(RAW_DATA_DIR / "bureau_data.csv")

print("Customers:", customers.shape)
print("Loans:", loans.shape)
print("Bureau:", bureau.shape)


Customers: (50000, 12)
Loans: (50000, 15)
Bureau: (50000, 8)


In [3]:

# Check the loan default target distribution

target_summary = (
    loans["default"]
    .value_counts(dropna=False)
    .rename_axis("default")
    .to_frame("records")
)

target_summary["percentage"] = (
    target_summary["records"] / len(loans) * 100
).round(3)

display(target_summary)


,records,percentage
default,,
False,45703,91.406
True,4297,8.594


In [4]:

# Inspect the schema and data quality of each dataset

for name, df in [
    ("customers", customers),
    ("loans", loans),
    ("bureau", bureau),
]:
    print(f"\n{'=' * 60}")
    print(f"{name.upper()} SCHEMA")
    print("=" * 60)

    schema = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing": df.isna().sum().values,
        "unique_values": df.nunique(dropna=False).values,
    })

    display(schema)



CUSTOMERS SCHEMA


,column,dtype,missing,unique_values
0,cust_id,str,0,50000
1,age,int64,0,53
2,gender,str,0,2
3,marital_status,str,0,2
4,employment_status,str,0,2
5,income,int64,0,8745
6,number_of_dependants,int64,0,6
7,residence_type,str,62,4
8,years_at_current_address,int64,0,31
9,city,str,0,10



LOANS SCHEMA


,column,dtype,missing,unique_values
0,loan_id,str,0,50000
1,cust_id,str,0,50000
2,loan_purpose,str,0,5
3,loan_type,str,0,2
4,sanction_amount,int64,0,13012
5,loan_amount,int64,0,12282
6,processing_fee,float64,0,12289
7,gst,int64,0,12282
8,net_disbursement,int64,0,12282
9,loan_tenure_months,int64,0,54



BUREAU SCHEMA


,column,dtype,missing,unique_values
0,cust_id,str,0,50000
1,number_of_open_accounts,int64,0,4
2,number_of_closed_accounts,int64,0,3
3,total_loan_months,int64,0,222
4,delinquent_months,int64,0,25
5,total_dpd,int64,0,168
6,enquiry_count,int64,0,9
7,credit_utilization_ratio,int64,0,100


In [5]:

# Numerical feature distributions and skewness

numeric_datasets = {
    "customers": customers,
    "loans": loans,
    "bureau": bureau,
}

for name, df in numeric_datasets.items():
    numeric = df.select_dtypes(include="number")

    print(f"\n{'=' * 65}")
    print(f"{name.upper()}: NUMERICAL DISTRIBUTIONS")
    print("=" * 65)

    summary = numeric.describe().T
    summary["skewness"] = numeric.skew()
    summary["zero_count"] = (numeric == 0).sum()
    summary["negative_count"] = (numeric < 0).sum()

    display(summary.round(2))



CUSTOMERS: NUMERICAL DISTRIBUTIONS


,count,mean,std,min,25%,50%,75%,max,skewness,zero_count,negative_count
age,50000.0,39.55,9.85,18.0,33.0,40.0,46.0,70.0,0.08,0,0
income,50000.0,2640897.50,2629441.06,0.0,803000.0,1892000.0,3332250.0,11999000.0,1.77,8,0
number_of_dependants,50000.0,1.94,1.54,0.0,0.0,2.0,3.0,5.0,0.15,13683,0
years_at_current_address,50000.0,16.02,8.93,1.0,8.0,16.0,24.0,31.0,-0.00,0,0
zipcode,50000.0,419090.86,168982.67,110001.0,302001.0,400001.0,560001.0,700001.0,-0.14,0,0



LOANS: NUMERICAL DISTRIBUTIONS


,count,mean,std,min,25%,50%,75%,max,skewness,zero_count,negative_count
sanction_amount,50000.0,4704827.52,6267276.39,0.0,1147000.0,2656000.0,5172250.0,52175000.0,3.21,8,0
loan_amount,50000.0,3999678.80,5376552.03,0.0,967000.0,2240000.0,4611000.0,47819000.0,3.28,8,0
processing_fee,50000.0,80494.71,117312.26,0.0,19340.0,44800.0,92240.0,5698029.9,9.26,8,0
gst,50000.0,719942.18,967779.37,0.0,174060.0,403200.0,829980.0,8607420.0,3.28,8,0
net_disbursement,50000.0,3199743.04,4301241.63,0.0,773600.0,1792000.0,3688800.0,38255200.0,3.28,8,0
loan_tenure_months,50000.0,25.94,12.43,6.0,16.0,24.0,35.0,59.0,0.42,0,0
principal_outstanding,50000.0,1337636.04,1214713.89,-1.0,424799.0,1002309.0,1790751.0,5000000.0,1.48,0,8
bank_balance_at_application,50000.0,989126.38,1046071.89,0.0,287007.5,668525.0,1238667.0,7846643.0,2.12,8,0



BUREAU: NUMERICAL DISTRIBUTIONS


,count,mean,std,min,25%,50%,75%,max,skewness,zero_count,negative_count
number_of_open_accounts,50000.0,2.50,1.12,1.0,1.0,3.0,4.0,4.0,-0.00,0,0
number_of_closed_accounts,50000.0,1.00,0.81,0.0,0.0,1.0,2.0,2.0,-0.00,16543,0
total_loan_months,50000.0,76.13,43.76,1.0,42.0,71.0,107.0,223.0,0.42,0,0
delinquent_months,50000.0,4.88,5.85,0.0,0.0,3.0,9.0,24.0,1.24,19998,0
total_dpd,50000.0,26.86,32.83,0.0,0.0,14.0,46.0,171.0,1.30,19998,0
enquiry_count,50000.0,5.01,2.03,1.0,4.0,5.0,6.0,9.0,-0.01,0,0
credit_utilization_ratio,50000.0,43.36,29.36,0.0,18.0,39.0,67.0,99.0,0.34,747,0


In [6]:

# Investigate suspicious loan amounts without modifying raw data

amount_columns = [
    "sanction_amount",
    "loan_amount",
    "processing_fee",
    "gst",
    "net_disbursement",
    "bank_balance_at_application",
]

print("ZERO COUNTS")
display(
    (loans[amount_columns] == 0)
    .sum()
    .rename("zero_count")
    .to_frame()
)

print("\nRECORDS WITH ZERO LOAN AMOUNT")
zero_loan_records = loans.loc[
    loans["loan_amount"] == 0,
    [
        "loan_id",
        "cust_id",
        *amount_columns,
        "default",
    ],
]

display(zero_loan_records)

print("\nRECORDS WITH NEGATIVE OUTSTANDING BALANCE")
negative_balance_records = loans.loc[
    loans["principal_outstanding"] < 0,
    [
        "loan_id",
        "cust_id",
        "loan_amount",
        "principal_outstanding",
        "default",
    ],
]

display(negative_balance_records)


ZERO COUNTS


,zero_count
sanction_amount,8
loan_amount,8
processing_fee,8
gst,8
net_disbursement,8
bank_balance_at_application,8



RECORDS WITH ZERO LOAN AMOUNT


,loan_id,cust_id,sanction_amount,loan_amount,processing_fee,gst,net_disbursement,bank_balance_at_application,default
2278,L02279,C02279,0,0,0.0,0,0,0,False
4950,L04951,C04951,0,0,0.0,0,0,0,False
11092,L11093,C11093,0,0,0.0,0,0,0,False
13356,L13357,C13357,0,0,0.0,0,0,0,False
27457,L27458,C27458,0,0,0.0,0,0,0,False
45224,L45225,C45225,0,0,0.0,0,0,0,False
46640,L46641,C46641,0,0,0.0,0,0,0,False
47428,L47429,C47429,0,0,0.0,0,0,0,False



RECORDS WITH NEGATIVE OUTSTANDING BALANCE


,loan_id,cust_id,loan_amount,principal_outstanding,default
2278,L02279,C02279,0,-1,False
4950,L04951,C04951,0,-1,False
11092,L11093,C11093,0,-1,False
13356,L13357,C13357,0,-1,False
27457,L27458,C27458,0,-1,False
45224,L45225,C45225,0,-1,False
46640,L46641,C46641,0,-1,False
47428,L47429,C47429,0,-1,False


In [9]:

from plotly.subplots import make_subplots
import plotly.graph_objects as go

financial_features = [
    "loan_amount",
    "sanction_amount",
    "processing_fee",
    "principal_outstanding",
]

titles = [
    "Loan Amount",
    "Sanction Amount",
    "Processing Fee",
    "Principal Outstanding",
]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=titles,
    vertical_spacing=0.18,
    horizontal_spacing=0.12,
)

for i, (column, title) in enumerate(
    zip(financial_features, titles)
):
    row = i // 2 + 1
    col = i % 2 + 1

    fig.add_trace(
        go.Histogram(
            x=loans[column],
            nbinsx=50,
            name=title,
            showlegend=False,
            hovertemplate=(
                "Value: %{x:,.0f}"
                "<br>Number of loans: %{y}"
                "<extra></extra>"
            ),
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(
        title_text=title,
        row=row,
        col=col,
    )

    fig.update_yaxes(
        title_text="Number of Loans",
        row=row,
        col=col,
    )

fig.update_layout(
    title="AtliQ Credit Intelligence: Financial Distributions",
    template="plotly_white",
    height=650,
    autosize=True,
    margin=dict(l=55, r=25, t=85, b=50),
)

fig.show(
    config={"responsive": True}
)


In [11]:

# Investigate the maximum principal outstanding value

max_outstanding = loans["principal_outstanding"].max()

print("Maximum principal outstanding:", max_outstanding)
print(
    "Records at maximum:",
    (loans["principal_outstanding"] == max_outstanding).sum(),
)

display(
    loans.loc[
        loans["principal_outstanding"] == max_outstanding,
        [
            "loan_id",
            "loan_amount",
            "sanction_amount",
            "principal_outstanding",
            "default",
        ],
    ].head(10)
)


Maximum principal outstanding: 5000000
Records at maximum: 1840


,loan_id,loan_amount,sanction_amount,principal_outstanding,default
20,L00021,25489000,26933000,5000000,False
44,L00045,7623000,10000000,5000000,False
99,L00100,34579000,34913000,5000000,True
102,L00103,19223000,20000000,5000000,True
111,L00112,7401000,10000000,5000000,False
143,L00144,31554000,34140000,5000000,False
146,L00147,14018000,20000000,5000000,False
178,L00179,29910000,32635000,5000000,False
185,L00186,25324000,33191000,5000000,False
230,L00231,9525000,10000000,5000000,False


In [12]:

# Investigate the concentration at the maximum value

MAX_OUTSTANDING = loans["principal_outstanding"].max()

at_max = loans.loc[
    loans["principal_outstanding"] == MAX_OUTSTANDING
].copy()

print(f"Maximum value: {MAX_OUTSTANDING:,.0f}")
print(f"Records at maximum: {len(at_max):,}")
print(f"Share of all loans: {len(at_max) / len(loans) * 100:.2f}%")

print("\nDefault distribution among records at maximum:")
display(
    at_max["default"]
    .value_counts(dropna=False)
    .rename("records")
    .to_frame()
)

print("\nSummary of original loan amounts:")
display(
    at_max[
        ["loan_amount", "sanction_amount", "principal_outstanding"]
    ].describe().round(2)
)

print("\nCompare records at the maximum with all other records:")
comparison = (
    loans.assign(
        at_maximum=loans["principal_outstanding"] == MAX_OUTSTANDING
    )
    .groupby("at_maximum")
    .agg(
        records=("loan_id", "size"),
        mean_loan_amount=("loan_amount", "mean"),
        median_loan_amount=("loan_amount", "median"),
        default_rate=("default", "mean"),
    )
)

comparison["default_rate_pct"] = (
    comparison["default_rate"] * 100
).round(2)

display(comparison.round(2))


Maximum value: 5,000,000
Records at maximum: 1,840
Share of all loans: 3.68%

Default distribution among records at maximum:


,records
default,
False,1671
True,169



Summary of original loan amounts:


,loan_amount,sanction_amount,principal_outstanding
count,1840.00,1840.00,1840.0
mean,19527519.02,22911546.20,5000000.0
std,10248770.88,11779789.16,0.0
min,6959000.00,7814000.00,5000000.0
25%,9469500.00,10000000.00,5000000.0
50%,17406000.00,20000000.00,5000000.0
75%,28403000.00,33321250.00,5000000.0
max,47667000.00,52175000.00,5000000.0



Compare records at the maximum with all other records:


,records,mean_loan_amount,median_loan_amount,default_rate,default_rate_pct
at_maximum,,,,,
False,48160,3406422.45,2131000.0,0.09,8.57
True,1840,19527519.02,17406000.0,0.09,9.18


In [13]:

# Investigate the upper boundary of principal_outstanding.
# This cell only reads and summarizes data.

outstanding_counts = (
    loans["principal_outstanding"]
    .value_counts()
    .sort_index()
)

print("Largest 15 distinct outstanding values by magnitude:")
display(
    outstanding_counts.tail(15)
    .rename_axis("principal_outstanding")
    .reset_index(name="records")
    .sort_values("principal_outstanding", ascending=False)
)

print("\nMost frequent outstanding values:")
display(
    loans["principal_outstanding"]
    .value_counts()
    .head(15)
    .rename_axis("principal_outstanding")
    .reset_index(name="records")
)

print("\nOutstanding balance by loan-amount range:")
loans["loan_amount_group"] = pd.cut(
    loans["loan_amount"],
    bins=[-1, 1_000_000, 5_000_000, 10_000_000, float("inf")],
    labels=[
        "Up to 10 lakh",
        "10 lakh to 50 lakh",
        "50 lakh to 1 crore",
        "Above 1 crore",
    ],
)

display(
    loans.groupby("loan_amount_group", observed=False)
    .agg(
        records=("loan_id", "size"),
        max_outstanding=("principal_outstanding", "max"),
        records_at_5m=(
            "principal_outstanding",
            lambda s: (s == 5_000_000).sum(),
        ),
        median_outstanding=("principal_outstanding", "median"),
    )
    .reset_index()
)


Largest 15 distinct outstanding values by magnitude:


,principal_outstanding,records
14,5000000,1840
13,4999347,1
12,4998902,1
11,4998271,1
10,4996350,1
9,4995176,1
8,4994816,1
7,4994749,1
6,4991886,1
5,4991806,1



Most frequent outstanding values:


,principal_outstanding,records
0,5000000,1840
1,190799,16
2,217439,14
3,130319,14
4,161279,14
5,81359,14
6,308879,14
7,142559,13
8,411839,13
9,277199,13



Outstanding balance by loan-amount range:


,loan_amount_group,records,max_outstanding,records_at_5m,median_outstanding
0,Up to 10 lakh,12909,719999,0,264374.0
1,10 lakh to 50 lakh,26016,3599279,0,1102947.0
2,50 lakh to 1 crore,6953,5000000,577,1746946.0
3,Above 1 crore,4122,5000000,1263,3811748.5


In [14]:

# Investigate relationships among financial fields.
# Uses a separate copy for analysis; raw CSV files remain unchanged.

financial = loans[
    [
        "loan_id",
        "loan_amount",
        "sanction_amount",
        "principal_outstanding",
        "processing_fee",
        "gst",
        "net_disbursement",
        "bank_balance_at_application",
        "default",
    ]
].copy()

financial["outstanding_to_loan_ratio"] = (
    financial["principal_outstanding"]
    / financial["loan_amount"].replace(0, float("nan"))
)

financial["sanction_minus_loan"] = (
    financial["sanction_amount"] - financial["loan_amount"]
)

financial["loan_minus_disbursement"] = (
    financial["loan_amount"] - financial["net_disbursement"]
)

financial["outstanding_exceeds_loan"] = (
    financial["principal_outstanding"] > financial["loan_amount"]
)

print("Financial relationship checks")
print("-" * 55)

print(
    "Outstanding exceeds loan amount:",
    int(financial["outstanding_exceeds_loan"].sum()),
)

print(
    "Sanction amount below loan amount:",
    int((financial["sanction_amount"] < financial["loan_amount"]).sum()),
)

print(
    "Net disbursement exceeds loan amount:",
    int((financial["net_disbursement"] > financial["loan_amount"]).sum()),
)

print(
    "Negative loan-minus-disbursement difference:",
    int((financial["loan_minus_disbursement"] < 0).sum()),
)

print("\nSummary of financial differences:")
display(
    financial[
        [
            "sanction_minus_loan",
            "loan_minus_disbursement",
            "outstanding_to_loan_ratio",
        ]
    ].describe().round(3)
)

print("\nExamples where outstanding exceeds original loan amount:")
display(
    financial.loc[
        financial["outstanding_exceeds_loan"],
        [
            "loan_id",
            "loan_amount",
            "sanction_amount",
            "principal_outstanding",
            "default",
        ],
    ].head(10)
)


Financial relationship checks
-------------------------------------------------------
Outstanding exceeds loan amount: 0
Sanction amount below loan amount: 0
Net disbursement exceeds loan amount: 0
Negative loan-minus-disbursement difference: 0

Summary of financial differences:


,sanction_minus_loan,loan_minus_disbursement,outstanding_to_loan_ratio
count,5.000000e+04,50000.000,49992.000
mean,7.051487e+05,799935.760,0.467
std,1.151264e+06,1075310.407,0.215
min,0.000000e+00,0.000,0.082
25%,1.090000e+05,193400.000,0.249
50%,3.190000e+05,448000.000,0.483
75%,8.020000e+05,922200.000,0.720
max,1.423000e+07,9563800.000,0.720



Examples where outstanding exceeds original loan amount:


,loan_id,loan_amount,sanction_amount,principal_outstanding,default


In [15]:

# Check possible financial formula relationships.
# No raw data is modified.

financial = loans[
    [
        "loan_id",
        "loan_amount",
        "sanction_amount",
        "processing_fee",
        "gst",
        "net_disbursement",
        "default",
    ]
].copy()

financial["expected_gst_18pct"] = (
    financial["processing_fee"] * 0.18
)

financial["gst_difference"] = (
    financial["gst"] - financial["expected_gst_18pct"]
)

financial["expected_net_disbursement"] = (
    financial["loan_amount"]
    - financial["processing_fee"]
    - financial["gst"]
)

financial["net_disbursement_difference"] = (
    financial["net_disbursement"]
    - financial["expected_net_disbursement"]
)

print("GST formula investigation")
print("Absolute GST difference, in rupees:")
display(financial["gst_difference"].abs().describe().round(2))

print("\nNet disbursement formula investigation")
print("Absolute net-disbursement difference, in rupees:")
display(
    financial["net_disbursement_difference"]
    .abs()
    .describe()
    .round(2)
)

print("\nTen largest GST discrepancies:")
display(
    financial.assign(
        absolute_gst_difference=financial["gst_difference"].abs()
    )
    .nlargest(10, "absolute_gst_difference")[
        [
            "loan_id",
            "processing_fee",
            "gst",
            "expected_gst_18pct",
            "gst_difference",
        ]
    ]
)


GST formula investigation
Absolute GST difference, in rupees:


count      50000.00
mean      705495.03
std       948441.07
min            0.00
25%       170402.40
50%       395047.80
75%       813380.40
max      8435271.60
Name: gst_difference, dtype: float64


Net disbursement formula investigation
Absolute net-disbursement difference, in rupees:


count      50000.00
mean         501.13
std        47031.80
min            0.00
25%            0.00
50%            0.00
75%            0.00
max      5605249.90
Name: net_disbursement_difference, dtype: float64


Ten largest GST discrepancies:


,loan_id,processing_fee,gst,expected_gst_18pct,gst_difference
49413,L49414,956380.0,8607420,172148.4,8435271.6
47967,L47968,953340.0,8580060,171601.2,8408458.8
13905,L13906,943460.0,8491140,169822.8,8321317.2
32895,L32896,934400.0,8409600,168192.0,8241408.0
22367,L22368,931560.0,8384040,167680.8,8216359.2
32172,L32173,921720.0,8295480,165909.6,8129570.4
10938,L10939,918300.0,8264700,165294.0,8099406.0
33353,L33354,900720.0,8106480,162129.6,7944350.4
48802,L48803,876680.0,7890120,157802.4,7732317.6
49184,L49185,876280.0,7886520,157730.4,7728789.6


In [16]:

# Quantify financial formula matches and exceptions.

net_diff = financial["net_disbursement_difference"]
gst_diff = financial["gst_difference"]

print("NET DISBURSEMENT FORMULA")
print("-" * 45)
print("Exact matches:", int(net_diff.eq(0).sum()))
print("Nonzero differences:", int(net_diff.ne(0).sum()))
print(
    "Percentage exact:",
    round(net_diff.eq(0).mean() * 100, 3),
)

print("\nGST AT 18% OF PROCESSING FEE")
print("-" * 45)
print("Exact matches:", int(gst_diff.eq(0).sum()))
print("Nonzero differences:", int(gst_diff.ne(0).sum()))

print("\nLargest net-disbursement exceptions:")
display(
    financial.assign(
        absolute_net_difference=net_diff.abs()
    )
    .nlargest(10, "absolute_net_difference")[
        [
            "loan_id",
            "loan_amount",
            "processing_fee",
            "gst",
            "net_disbursement",
            "net_disbursement_difference",
            "default",
        ]
    ]
)

print("\nLargest net-disbursement exceptions: target distribution")
largest_exceptions = financial.loc[
    net_diff.abs().nlargest(100).index
]

display(
    largest_exceptions["default"]
    .value_counts(dropna=False)
    .rename("records")
    .to_frame()
)


NET DISBURSEMENT FORMULA
---------------------------------------------
Exact matches: 49993
Nonzero differences: 7
Percentage exact: 99.986

GST AT 18% OF PROCESSING FEE
---------------------------------------------
Exact matches: 8
Nonzero differences: 49992

Largest net-disbursement exceptions:


,loan_id,loan_amount,processing_fee,gst,net_disbursement,net_disbursement_difference,default
12294,L12295,4639000,5.698030e+06,835020,3711200,5.605250e+06,False
33705,L33706,3812000,5.678021e+06,686160,3049600,5.601781e+06,False
9898,L09899,3626000,5.293544e+06,652680,2900800,5.221024e+06,False
29305,L29306,2616000,3.036378e+06,470880,2092800,2.984058e+06,False
23981,L23982,2234000,2.669791e+06,402120,1787200,2.625111e+06,False
47089,L47090,1738000,1.858965e+06,312840,1390400,1.824205e+06,False
28174,L28175,966000,1.214493e+06,173880,772800,1.195173e+06,False
0,L00001,2467000,4.934000e+04,444060,1973600,0.000000e+00,False
1,L00002,3883000,7.766000e+04,698940,3106400,0.000000e+00,False
2,L00003,2170000,4.340000e+04,390600,1736000,0.000000e+00,False



Largest net-disbursement exceptions: target distribution


,records
default,
False,91
True,9


In [17]:

# Investigate all nonzero net-disbursement differences.
# The source CSV files remain unchanged.

exceptions = financial.loc[
    financial["net_disbursement_difference"] != 0
].copy()

exceptions["absolute_difference"] = (
    exceptions["net_disbursement_difference"].abs()
)

exceptions["fee_exceeds_loan"] = (
    exceptions["processing_fee"] > exceptions["loan_amount"]
)

print("Number of formula exceptions:", len(exceptions))

print("\nAll formula exceptions:")
display(
    exceptions[
        [
            "loan_id",
            "loan_amount",
            "processing_fee",
            "gst",
            "net_disbursement",
            "net_disbursement_difference",
            "absolute_difference",
            "fee_exceeds_loan",
            "default",
        ]
    ].sort_values("absolute_difference", ascending=False)
)

print("\nException summary:")
display(
    exceptions[
        [
            "loan_amount",
            "processing_fee",
            "gst",
            "net_disbursement",
            "absolute_difference",
        ]
    ].describe().round(2)
)

print("\nTarget distribution among the seven exceptions:")
display(
    exceptions["default"]
    .value_counts(dropna=False)
    .rename("records")
    .to_frame()
)


Number of formula exceptions: 7

All formula exceptions:


,loan_id,loan_amount,processing_fee,gst,net_disbursement,net_disbursement_difference,absolute_difference,fee_exceeds_loan,default
12294,L12295,4639000,5.698030e+06,835020,3711200,5.605250e+06,5.605250e+06,True,False
33705,L33706,3812000,5.678021e+06,686160,3049600,5.601781e+06,5.601781e+06,True,False
9898,L09899,3626000,5.293544e+06,652680,2900800,5.221024e+06,5.221024e+06,True,False
29305,L29306,2616000,3.036378e+06,470880,2092800,2.984058e+06,2.984058e+06,True,False
23981,L23982,2234000,2.669791e+06,402120,1787200,2.625111e+06,2.625111e+06,True,False
47089,L47090,1738000,1.858965e+06,312840,1390400,1.824205e+06,1.824205e+06,True,False
28174,L28175,966000,1.214493e+06,173880,772800,1.195173e+06,1.195173e+06,True,False



Exception summary:


,loan_amount,processing_fee,gst,net_disbursement,absolute_difference
count,7.00,7.00,7.00,7.00,7.00
mean,2804428.57,3635602.96,504797.14,2243542.86,3579514.39
std,1286715.18,1892261.62,231608.73,1029372.14,1867233.68
min,966000.00,1214492.67,173880.00,772800.00,1195172.67
25%,1986000.00,2264377.90,357480.00,1588800.00,2224657.90
50%,2616000.00,3036378.01,470880.00,2092800.00,2984058.01
75%,3719000.00,5485782.19,669420.00,2975200.00,5411402.19
max,4639000.00,5698029.90,835020.00,3711200.00,5605249.90



Target distribution among the seven exceptions:


,records
default,
False,7


### Finding: Net-disbursement formula exceptions

The tested relationship `net_disbursement = loan_amount - processing_fee - gst` matched 49,993 of 50,000 records (99.986%).

Seven records had nonzero differences. All seven shared the following characteristics:

- `processing_fee` exceeded `loan_amount`.
- All seven were labelled `default=False`.
- The absolute formula discrepancies ranged from ₹11,95,172.67 to ₹56,05,249.90.

The GST hypothesis `gst = 0.18 × processing_fee` was not supported as a general rule: only 8 records matched it exactly.

**Interpretation:** the seven records are financial data-quality exceptions requiring further investigation. The intended field definitions and the reason for these exceptions remain undocumented.

**Action:** preserve the raw records, retain the anomaly flags, and investigate the source-generation rules before deciding whether any correction is justified.

In [19]:

# Inspect the seven exceptions across additional loan fields.
# Read-only analysis using the existing DataFrame.

exception_ids = exceptions["loan_id"].tolist()

exception_details = loans.loc[
    loans["loan_id"].isin(exception_ids),
    [
        "loan_id",
        "cust_id",
        "sanction_amount",
        "loan_amount",
        "processing_fee",
        "gst",
        "net_disbursement",
        "principal_outstanding",
        "bank_balance_at_application",
        "loan_tenure_months",
        "default",
    ],
].copy()

display(
    exception_details.sort_values("loan_id").reset_index(drop=True)
)

print("\nSanction amount versus loan amount:")
print(
    "All sanction amounts >= loan amounts:",
    bool(
        (
            exception_details["sanction_amount"]
            >= exception_details["loan_amount"]
        ).all()
    ),
)

print("\nPrincipal outstanding versus loan amount:")
print(
    "All outstanding balances <= loan amounts:",
    bool(
        (
            exception_details["principal_outstanding"]
            <= exception_details["loan_amount"]
        ).all()
    ),
)


,loan_id,cust_id,sanction_amount,loan_amount,processing_fee,gst,net_disbursement,principal_outstanding,bank_balance_at_application,loan_tenure_months,default
0,L09899,C09899,5000000,3626000,5.293544e+06,652680,2900800,2610719,2849384,6,False
1,L12295,C12295,5312000,4639000,5.698030e+06,835020,3711200,1890831,1282049,25,False
2,L23982,C23982,2649000,2234000,2.669791e+06,402120,1787200,920114,511339,32,False
3,L28175,C28175,1025000,966000,1.214493e+06,173880,772800,695519,390870,18,False
4,L29306,C29306,2872000,2616000,3.036378e+06,470880,2092800,1883519,999324,21,False
5,L33706,C33706,5000000,3812000,5.678021e+06,686160,3049600,2744639,4612551,9,False
6,L47090,C47090,1815000,1738000,1.858965e+06,312840,1390400,1032442,626462,18,False



Sanction amount versus loan amount:
All sanction amounts >= loan amounts: True

Principal outstanding versus loan amount:
All outstanding balances <= loan amounts: True


### Finding: Net-Disbursement Formula Exceptions

**Observed evidence**

- Seven records violate the tested formula:

  `net_disbursement = loan_amount - processing_fee - gst`

- In all seven records, `processing_fee` exceeds `loan_amount`.
- All seven records have `default = False`.
- The absolute formula discrepancies range from ₹11,95,172.67 to ₹56,05,249.90.
- All seven records satisfy `sanction_amount >= loan_amount`.
- All seven records satisfy `principal_outstanding <= loan_amount`.

**Interpretation**

These records are financial data-quality exceptions requiring investigation. The processing fees are unusually large relative to their corresponding loan amounts. However, the dataset does not provide sufficient documentation to establish whether the values are incorrect, generated anomalies, or valid exceptions.

**Action**

- Preserve the original records.
- Retain anomaly flags for subsequent analysis.
- Investigate the source and meaning of the unusually large processing fees.
- Do not silently correct, delete, or replace these values during exploratory analysis.

**Limitation**

The tested financial relationships are hypotheses based on the available data, not confirmed business rules.

# Day 2: Key Findings and EDA Conclusion

## 1. Dataset overview

- Three datasets contain 50,000 records each: customers, loans, and bureau data.
- The merged analysis table contains 50,000 rows and 33 columns.
- The loan default target contains 4,297 defaults (8.594%) and 45,703 non-defaults (91.406%).

## 2. Key observations

- **Loan purpose:** Home loans have an observed default rate of 15.52%, compared with 4.54% for Personal loans and 4.29% for Auto loans.
- **Residence type:** Rented residences have an observed default rate of 16.34%, compared with 5.57% for Owned residences.
- **Loan type:** Secured loans have an observed default rate of 10.77%, compared with 4.54% for Unsecured loans.
- **Financial distributions:** Loan amounts, sanction amounts, and processing fees are strongly right-skewed, with a small number of very large values.
- **Missing and inconsistent data:** The customer dataset contains 62 missing residence-type values, and the loan-purpose field contains 22 records labelled `Personaal`.
- **Financial anomalies:** Eight records have zero loan amounts and an outstanding balance of -1. Seven additional records violate the tested net-disbursement formula because their processing fees exceed their loan amounts.

## 3. Data-quality decisions

- Preserve the original CSV files unchanged.
- Document missing values, inconsistent category labels, and suspicious financial values.
- Investigate anomalies before deciding whether they represent valid observations or data errors.
- Avoid silently removing or correcting records.

## 4. Important limitations

The dataset documentation does not establish the target-generation process, observation horizon, or all applicable financial business rules.

The observed default-rate differences are descriptive associations, not evidence of causation or proof of predictive performance. Feature availability and potential target leakage must be investigated before model development.

## 5. Day 2 conclusion

Initial exploratory data analysis has established the dataset structure, target distribution, categorical default-rate differences, numerical distributions, and several financial data-quality concerns.

The next phase will investigate feature suitability, relationships between variables, and leakage risks before any model is trained.